# Assignment 2, Tori Healey

The entirety of this notebook could be considered "Task 3" of the assignment -- creating an end to end ML pipeline. In this notebook, after importing helper functions and configuring paths, I run the major steps on a ML pipeline, including:
- Data ingestion
- Data preprocessing
- Feature creation
    - Includes feature store usage
- Training a model & generating predictions (logging using MLflow)
- Evaluating a model (logging using MLflow)

To replicate this notebook on your own environment, change the key variables in the configuration cell (cell 5).

In [0]:
# To be run if necessary
#%pip install databricks-feature-engineering
#dbutils.library.restartPython()

In [0]:
# Import standard packages
import mlflow
import mlflow.sklearn
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
import sys, os
# Import helper functions that I wrote to conduct EDA, do data preprocessing, run model, etc.
sys.path.append(os.path.abspath("../src"))
from pipeline_helper_functions import *

In [0]:
# Configure: change the following file paths if necessary
CATALOG = "workspace"
SCHEMA = "athletes"
DATA_PATH = f"/Volumes/{CATALOG}/default/{SCHEMA}/athletes.csv"
# MLflow path
experiment_name = "/Users/healeyv@uchicago.edu/healey_assignment_2_experiments"


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

### Task 1: Ingest data, perform EDA

In [0]:
# Ingest data
data = pd.read_csv(DATA_PATH)
# Validate that data loaded in by checking the length
len(data)

Perform EDA using helper functions from pipeline_helper_functions.py.

In [0]:
# Prints list of data types by feature
# Also prints basic summary statistics (count, mean, max, etc) for numerical vars
eda_summary(data)

In [0]:
# Returns the number of missing values by feature
eda_missingness(data)

In [0]:
# Plots histograms for numerical features to show distribution of data
eda_numer_distribution(data)

In [0]:
# Prints information on categorical variables, such as the number of unique values that feature has and some examples of values the feature takes on
# This will be helpful in deciding how to encode categorical variables and deciding which ones to drop
eda_categorical_exploration(data)

This preliminary EDA highlights a few things about the data set:
- it has a large number of missing values: many features are missing for upwards of 75% of the data set
- many numerical features have large outlier values that seem impossible, which suggests we will need to remove them
- some categorical variables have many unique values, which will make them difficult to OneHotEncode. We will need to drop some of them that have a lot of potential values since these will not be useful (ie. team name)
- additionally, some categorical variables seem to take the form of a survey response in which the respondent has selected several answers. We could split this up into a series of dummy variables rather than one list.

Using these findings, let's preprocess the data.

The pipeline below uses the helper functions written in pipeline_helper_functions.py to clean the data. It takes these steps:
1. data_preprocessing: basic data cleaning function that removes outlier values, drops unhelpful variables, and drops missing values. This greatly reduces the size of the data set but makes what remains much easier to use in a ML model.
2. expand_multiselect: turns features that can take on several values (for example, where a respondent can select multiple answers) into a set of dummy variables that are 1 if the respondent selected that answer and 0 otherwise
3. create_dummies: turns the remaining categorical variables (like gender or region) into dummy variables
4. sanitize_column_names: cleans the column names that result from functions 2 and 3, since these are sometimes long or have strange characters in them

In [0]:
# Stack together helper functions using sklearn Pipeline
preprocessing_pipeline = Pipeline([
    ("data_preprocessing", FunctionTransformer(data_preprocessing)),
    ("expand_multiselect", FunctionTransformer(lambda df: expand_multiselect(df, columns=['eat', 'train', 'background', 'experience', 'schedule', 'howlong']))),
    ("create_dummies", FunctionTransformer(create_dummies)),
    ("sanitize_column_names", FunctionTransformer(sanitize_column_names)),
])

# run the Pipeline
processed_data = preprocessing_pipeline.fit_transform(data)

# verify that the data looks as desired
processed_data.head(10)


### Task 2: MLOps Platform selection & configuration

Since I conducted this analysis in DataBricks, I will use MLflow, which requires no major setup and is well-integrated with the DataBricks platform. This is part of the reason I chose to run this workflow in DataBricks -- it requires no additional setup to get MLflow running. MLflow will allow me to conduct the experiment logging/model tracking that this assignment requires in the later steps.

In [0]:
# set up MLflow
mlflow.set_experiment(experiment_name)

### Task 4 & 5: feature store integration; feature versioning

First, we will build two versions of the features. Both versions will iterate on the initial data preprocessing we did. This way, both versions of the data will be fairly clean, but they will have slightly different features.

Features V1: Does not expand upon the preprocessed data --> so, the basic preprocessing we did in task 1 (removing outliers, creating dummy variables) will remain, but no additional steps are taken or variables are created.

Features V2: Creates additional variables/interaction terms, such as height-to-weight ratio, speed to weight ratio, 5k to 400m speed ratio, and age bin instead of raw age.

NOTE: Both feature sets drop the components of the output feature (total_lift): candj, deadlift, snatch, and backsq. This will keep the model we run later from "cheating" its predictions.


In [0]:
# Build the new data sets using the build features helper functions
features_v1_df = build_features_v1(processed_data)
features_v2_df = build_features_v2(processed_data)
# Clean column names
features_v1_df = sanitize_column_names(features_v1_df)
features_v2_df = sanitize_column_names(features_v2_df)

# confirm that the build features functions and sanitizing column names functions worked as desired
features_v2_df.head(10)

Now, we will store the two feature definitions in the DataBricks native feature store (FeatureEngineeringClient).

In [0]:
# drop feature tables if they already exist
spark.sql("DROP TABLE IF EXISTS workspace.athletes.features_v1")
spark.sql("DROP TABLE IF EXISTS workspace.athletes.features_v2")

# create spark tables from the features data frames
features_v1_spark = spark.createDataFrame(features_v1_df)
features_v2_spark = spark.createDataFrame(features_v2_df)

# initialize FeatureEngineeringClient
fe = FeatureEngineeringClient()

# create FeatureEngineeringClient data feature version 1
fe.create_table(
    name="workspace.athletes.features_v1",
    primary_keys=["athlete_id"],
    df=features_v1_spark,
    description="v1: 'basic' form of the data set, no feature engineering done"
)

# create FeatureEngineeringClient data feature version 2
fe.create_table(
    name="workspace.athletes.features_v2",
    primary_keys=["athlete_id"],
    df=features_v2_spark,
    description="v2: 'advanced' form of the data set: target feature added, categorical vars converted to dummies, new features such as height to weight ratio, speed to weight ratio, 5k to 400m ratio, and age bin created"
)

I now demonstrate how features are stored within the pipeline and print the description/features of each feature table to confirm they are working as expected. In the print statement below, you can see how the two data sets have slightly different features, with version 2 having more features.

In [0]:
# Confirm feature tables are registered and inspect their metadata
v1_table_info = fe.get_table(name="workspace.athletes.features_v1")
v2_table_info = fe.get_table(name="workspace.athletes.features_v2")

print(v1_table_info.description, v1_table_info.features)
print(v2_table_info.description, v2_table_info.features)

Finally, I demonstrate how features are retrieved within the pipeline by creating the data sets we will use for model building using only the information from the feature store.

In [0]:
# labels_df: row ID and target feature
labels_df = processed_data[["athlete_id", "total_lift"]]
labels_spark = spark.createDataFrame(labels_df)

# Retrieve v1 features from feature store
training_set_v1 = fe.create_training_set(
    df=labels_spark,
    feature_lookups=[
        FeatureLookup(table_name="workspace.athletes.features_v1", lookup_key="athlete_id")
    ],
    label="total_lift",
    exclude_columns=["athlete_id"]  # drop the join key from training data
)
# Build v1 in Pandas
training_df_v1 = training_set_v1.load_df().toPandas()

# Retrieve v2 features from feature store
training_set_v2 = fe.create_training_set(
    df=labels_spark,
    feature_lookups=[
        FeatureLookup(table_name="workspace.athletes.features_v2", lookup_key="athlete_id")
    ],
    label="total_lift",
    exclude_columns=["athlete_id"]
)
# Build v2 in pandas
training_df_v2 = training_set_v2.load_df().toPandas()

# Confirm that the above code worked
training_df_v2.head(10)

### Task 6: Experimentation & Model Training

We conclude by running 4 experimental XGBoost models:
    
    1. Data version 1, 250 estimators with max depth of 5
    2. Data version 1, 500 estimators with max depth of 10
    3. Data version 2, 250 estimators with max depth of 5
    4. Data version 1, 500 estimators with max depth of 10

Each model, its hyperparameters, outputs/performance, and artifacts are logged in MLflow for later review.

This evaluation will help us assess if 1) engineering additional features improved predictive accuracy and 2) allowing for more "robust" hyperparameters (n_estimators=500 and max_depth=10) improved the model or led to overfitting.

** Please note that the code used to build the run_xgb function was based off of a lab covered in office hours and is not entirely original work.

In [0]:
# First, define one more function that does the last few preprocessing steps (splitting, scaling) and then training the XGBoost model.
def prepare_and_train(training_df, feature_version, target_col="total_lift", 
                       test_size=0.3, n_estimators=250, max_depth=5):
    X = training_df.drop(columns=[target_col])
    y = training_df[target_col]
    
    # set random seed = 42
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    X_train, X_test = scale_data(X_train, X_test)
    
    # run my predefined helper function
    run_xgb(X_train, X_test, y_train, y_test, 
            feature_version=feature_version, 
            n_estimators=n_estimators, max_depth=max_depth)

In [0]:
# Define the two versions of the data set
training_dfs = {1: training_df_v1, 2: training_df_v2}
# Define 2 hyperparameter configurations. We will play with commonly edited XGBoost hyperparameters: number of estimators and max depth
hyperparam_configs = [
    {"n_estimators": 250, "max_depth": 5},
    {"n_estimators": 500, "max_depth": 10},
]

# Run the model. prepare_and_train will output graphs indicating model performance. For full model performance, see the experiment in MLflow.
for version, training_df in training_dfs.items():
    for params in hyperparam_configs:
        prepare_and_train(training_df, feature_version=version, **params)

For a full analysis of experiment results, please see the document titled "Healey Experiment Comparison".